In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Load all results from previous weeks
df = pd.read_csv("../data/featured_sample.csv")
df['date'] = pd.to_datetime(df['date'])
inventory = pd.read_csv("../data/inventory_optimization.csv")
abc_analysis = pd.read_csv("../data/abc_inventory_analysis.csv")
model_comparison = pd.read_csv("../data/final_model_comparison.csv")

print("All data loaded successfully!")
print("Featured data:", df.shape)
print("Inventory:", inventory.shape)
print("ABC analysis:", abc_analysis.shape)
print("Model comparison:", model_comparison.shape)

All data loaded successfully!
Featured data: (500000, 32)
Inventory: (30490, 11)
ABC analysis: (30490, 16)
Model comparison: (5, 3)


In [2]:
# Generate Reorder Schedule for Next 30 Days
print("Generating reorder schedule...")

reorder_schedule = abc_analysis[abc_analysis['abc_class'].isin(['A', 'B'])].copy()
reorder_schedule = reorder_schedule.sort_values('reorder_point', ascending=False)

reorder_schedule['days_until_reorder'] = (
    reorder_schedule['reorder_point'] / (reorder_schedule['avg_daily_demand'] + 0.01)
).round(0)

reorder_schedule['priority'] = reorder_schedule['days_until_reorder'].apply(
    lambda x: 'Urgent' if x <= 7 else ('Soon' if x <= 14 else 'Normal')
)

print("Reorder Schedule Generated!")
print("\nPriority Distribution:")
print(reorder_schedule['priority'].value_counts())

print("\nTop 10 Urgent Reorders:")
print(reorder_schedule[reorder_schedule['priority'] == 'Urgent'].nsmallest(
    10, 'days_until_reorder')[
    ['item_id', 'store_id', 'cat_id', 'order_quantity', 'days_until_reorder']
])

Generating reorder schedule...
Reorder Schedule Generated!

Priority Distribution:
priority
Soon      15005
Urgent      224
Normal       16
Name: count, dtype: int64

Top 10 Urgent Reorders:
               item_id store_id     cat_id  order_quantity  days_until_reorder
10645  HOUSEHOLD_1_161     WI_2  HOUSEHOLD             6.0                 4.0
10606    HOBBIES_1_069     CA_2    HOBBIES             6.0                 4.0
12171    HOBBIES_1_200     TX_1    HOBBIES             6.0                 4.0
9978     HOBBIES_1_297     WI_3    HOBBIES             6.0                 4.0
12962    HOBBIES_1_166     WI_1    HOBBIES             6.0                 4.0
14510    HOBBIES_1_050     CA_2    HOBBIES             6.0                 4.0
14507  HOUSEHOLD_2_223     WI_2  HOUSEHOLD             6.0                 4.0
14667    HOBBIES_1_246     CA_2    HOBBIES             6.0                 4.0
8003   HOUSEHOLD_1_342     TX_2  HOUSEHOLD             6.0                 4.0
8992       FOODS_3_

In [ ]:
# Revenue Impact Forecast - Next 30 Days
print("Calculating revenue impact forecast...")

revenue_forecast = inventory.groupby('cat_id').agg(
    total_items=('item_id', 'count'),
    total_monthly_revenue=('monthly_revenue', 'sum'),
    avg_order_qty=('order_quantity', 'mean')
).reset_index()

revenue_forecast['revenue_pct'] = (
    revenue_forecast['total_monthly_revenue'] / 
    revenue_forecast['total_monthly_revenue'].sum() * 100
).round(2)

print("\n30-Day Revenue Forecast by Category:")
print(revenue_forecast)

# Plot
plt.figure(figsize=(10,5))
plt.pie(revenue_forecast['total_monthly_revenue'], 
        labels=revenue_forecast['cat_id'],
        autopct='%1.1f%%',
        colors=['#0891B2', '#7C3AED', '#059669'])
plt.title("30-Day Revenue Forecast by Category")
plt.show()

total_revenue = revenue_forecast['total_monthly_revenue'].sum()
print(f"\nTotal Projected 30-Day Revenue: ${total_revenue:,.2f}")